# ✅ Question 1: Environment Setup & API Key Security

**Question:** Why is it important to load API keys from environment variables instead of hardcoding them in your notebook? Write code to:
1. Load the OpenAI API key from a `.env` file
2. Raise a clear error if the key is missing
3. Verify the key is loaded successfully (without printing the actual key)

**Answer:**

**Why use environment variables:**
- **Security**: Prevents API keys from being exposed in source code
- **Version Control**: .env files are typically git-ignored, preventing accidental commits
- **Flexibility**: Easy to change keys across environments (dev/staging/prod)
- **Best Practice**: Industry standard for sensitive credentials management

In [ ]:
# Solution: Environment Setup with API Key Validation

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve the API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Validate that the key exists
if not OPENAI_API_KEY:
    raise ValueError(
        "❌ OPENAI_API_KEY not found!\n"
        "Please create a .env file with: OPENAI_API_KEY=your_key_here"
    )

# Verify the key is loaded (show only first/last 4 characters for security)
masked_key = f"{OPENAI_API_KEY[:4]}...{OPENAI_API_KEY[-4:]}"
print(f"✅ API Key loaded successfully: {masked_key}")
print(f"✅ Key length: {len(OPENAI_API_KEY)} characters")

---

# ✅ Question 2: PDF Loading and Text Extraction

**Question:** Write code to:
1. Find all PDF files in a `resumes` folder
2. Load each PDF using PyPDFLoader
3. Print the number of pages in each resume
4. Combine all pages into a single text string for each resume

In [ ]:
# Solution: PDF Loading and Multi-Page Handling

from langchain_community.document_loaders import PyPDFLoader
import os

# Step 1: Find all PDF files in the resumes folder
resume_folder = "resumes"

# Filter for .pdf files (case-insensitive)
resume_files = [
    f for f in os.listdir(resume_folder)
    if f.lower().endswith(".pdf")
]

print(f"📁 Found {len(resume_files)} PDF files:")
print(resume_files)
print("\n" + "="*60 + "\n")

# Step 2 & 3: Load each PDF and track pages
all_resumes = []

for file in resume_files:
    # Construct full file path
    path = os.path.join(resume_folder, file)
    
    # Load PDF using PyPDFLoader
    loader = PyPDFLoader(path)
    docs = loader.load()  # Returns list of Document objects (one per page)
    
    # Print page count
    print(f"📄 {file}: {len(docs)} page(s)")
    
    # Step 4: Combine all pages into single string
    # Each doc.page_content contains the text from one page
    combined_text = "\n\n".join([doc.page_content for doc in docs])
    
    # Store filename and text
    all_resumes.append({
        "filename": file,
        "docs": docs,
        "full_text": combined_text,
        "page_count": len(docs)
    })

print("\n✅ All PDFs loaded successfully!")
print(f"Total resumes processed: {len(all_resumes)}")

**Key Learnings:**
- PyPDFLoader returns a list of Document objects (one per page)
- Each Document has a `page_content` attribute containing the text
- Multi-page resumes need to be combined before processing
- List comprehension is efficient for filtering files

---

# ✅ Question 3: Text Cleaning with Regex

**Question:** PDF extraction often includes extra whitespace, newlines, and formatting artifacts. Write a function that:
1. Takes a list of Document objects
2. Combines all pages with double newlines
3. Uses regex to collapse multiple spaces into one
4. Returns clean, trimmed text

In [ ]:
# Solution: Text Cleaning Function

import re

def combine_and_clean(docs):
    """
    Combine multiple Document objects and clean the text.
    
    Args:
        docs (list): List of Document objects from PyPDFLoader
    
    Returns:
        str: Clean, combined text with normalized whitespace
    """
    # Step 1: Combine all pages with paragraph separation
    text = "\n\n".join([d.page_content for d in docs])
    
    # Step 2: Use regex to collapse multiple whitespace characters into single space
    # \s+ matches one or more whitespace (spaces, tabs, newlines)
    text = re.sub(r"\s+", " ", text)
    
    # Step 3: Remove leading/trailing whitespace
    text = text.strip()
    
    return text


# Test the function with example text
print("🧪 Testing text cleaning function:\n")

# Simulate messy PDF text
messy_text = """John    Doe


    Senior   Data   Engineer
    
Skills:  Python,     SQL,    AWS


Experience:   5+ years"""

print("BEFORE cleaning:")
print(repr(messy_text))
print("\n" + "="*60 + "\n")

# Apply regex cleaning
cleaned_text = re.sub(r"\s+", " ", messy_text).strip()

print("AFTER cleaning:")
print(repr(cleaned_text))
print("\n✅ Text is now clean and normalized!")

**Why is text cleaning important?**
- **Token efficiency**: Reduces unnecessary whitespace that costs tokens
- **Better parsing**: LLMs perform better on clean, normalized text
- **Consistency**: Standardizes input format across different PDF sources
- **Cost savings**: Fewer tokens = lower API costs

---

# ✅ Question 4: Pydantic Schema Design

**Question:** Design a Pydantic schema for a resume with the following requirements:
1. Required fields: name, email
2. Optional fields: phone, summary
3. List of skills (default to empty list)
4. Nested education list with: degree, institution, years, cgpa
5. Create a sample instance and convert it to JSON

In [ ]:
# Solution: Complete Pydantic Schema with Nested Models

from pydantic import BaseModel, Field
from typing import List, Optional

# Nested model for education entries
class EducationEntry(BaseModel):
    """
    Schema for a single education entry.
    All fields are optional since resumes vary in completeness.
    """
    degree: Optional[str] = None
    institution: Optional[str] = None
    years: Optional[str] = None
    cgpa: Optional[str] = None


# Main resume schema
class ResumeSchema(BaseModel):
    """
    Complete schema for extracted resume data.
    Uses Field(default_factory=list) for mutable default values.
    """
    # Required fields (must be present, but can be None)
    name: Optional[str] = None
    email: Optional[str] = None
    
    # Optional contact info
    phone: Optional[str] = None
    
    # Lists with default factories (prevents mutable default bug)
    skills: List[str] = Field(default_factory=list)
    
    # Summary/experience section
    experience_summary: Optional[str] = None
    
    # Nested list of education entries
    education: List[EducationEntry] = Field(default_factory=list)


# Create a sample resume instance
sample_resume = ResumeSchema(
    name="Jane Smith",
    email="jane.smith@email.com",
    phone="+1-555-9876",
    skills=["Python", "Machine Learning", "SQL", "Docker", "AWS", "PySpark"],
    experience_summary="Senior Data Engineer with 7 years of experience building scalable ETL pipelines and ML systems.",
    education=[
        EducationEntry(
            degree="Master of Science in Computer Science",
            institution="MIT",
            years="2014-2016",
            cgpa="3.9/4.0"
        ),
        EducationEntry(
            degree="Bachelor of Engineering",
            institution="Stanford University",
            years="2010-2014",
            cgpa="3.7/4.0"
        )
    ]
)

# Display as formatted JSON
print("📋 Sample Resume Schema Instance:\n")
print(sample_resume.model_dump_json(indent=2))

# Access fields programmatically
print("\n" + "="*60)
print(f"\n✅ Candidate: {sample_resume.name}")
print(f"✅ Skills count: {len(sample_resume.skills)}")
print(f"✅ Education entries: {len(sample_resume.education)}")

**Key Pydantic Concepts:**

1. **Optional vs Required**: Use `Optional[type]` or `type | None` for optional fields
2. **Default Factories**: Use `Field(default_factory=list)` for mutable defaults (lists, dicts)
3. **Nested Models**: Create separate classes for complex nested structures
4. **Validation**: Pydantic automatically validates types and structure
5. **JSON Export**: Use `model_dump_json()` to serialize to JSON

---

# ✅ Question 5: LCEL Chain Construction

**Question:** Create a LangChain LCEL pipeline that:
1. Takes a resume text as input
2. Uses a PromptTemplate to instruct the LLM to extract structured data
3. Uses the pipe operator (|) to chain prompt → LLM
4. Uses temperature=0 for deterministic extraction
5. Test the chain with sample resume text

In [ ]:
# Solution: LCEL Chain with PromptTemplate

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json

# Step 1: Initialize LLM with temperature=0 for deterministic output
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # No randomness - same input = same output
    api_key=OPENAI_API_KEY
)

# Step 2: Create a PromptTemplate
prompt_extract = PromptTemplate(
    input_variables=["resume_text"],  # Variables that will be filled in
    template="""
You are an expert resume parser. Extract the following information from the resume text below:

- name (string)
- email (string)
- phone (string)
- skills (array of strings)
- experience_summary (string)
- education (array of objects with: degree, institution, years, cgpa)

Return ONLY valid JSON with these exact field names. No explanations or markdown.

Resume:
{resume_text}
"""
)

# Step 3: Create LCEL chain using pipe operator
# The | operator chains components: prompt output flows into LLM input
extract_chain = prompt_extract | llm

print("✅ LCEL Chain created: prompt_extract | llm")
print("\nChain components:")
print(f"  1. PromptTemplate with variables: {prompt_extract.input_variables}")
print(f"  2. ChatOpenAI (model={llm.model_name}, temperature={llm.temperature})")

In [ ]:
# Step 4: Test the chain with sample resume text

sample_resume_text = """
MICHAEL CHEN
michael.chen@techmail.com | (555) 123-4567

SUMMARY
Senior Data Engineer with 6 years of experience building scalable data pipelines and ML infrastructure.

SKILLS
Python, SQL, PySpark, Apache Airflow, AWS (S3, EMR, Redshift), Docker, Kubernetes, Git, Pandas, NumPy

EDUCATION
Master of Science in Data Science
University of California, Berkeley | 2015-2017 | GPA: 3.85/4.0

Bachelor of Technology in Computer Engineering
IIT Delhi | 2011-2015 | GPA: 8.7/10.0
"""

# Invoke the chain
print("🔄 Running extraction chain...\n")
result = extract_chain.invoke({"resume_text": sample_resume_text})

# Display the result
print("📊 Extraction Result:")
print("=" * 60)
print(result.content)
print("=" * 60)

# Parse and validate JSON
try:
    # Clean potential code fences
    import re
    content = result.content.strip()
    if content.startswith("```"):
        content = re.sub(r"```json\s*|```\s*", "", content).strip()
    
    parsed = json.loads(content)
    print("\n✅ Valid JSON extracted!")
    print(f"   Candidate: {parsed.get('name')}")
    print(f"   Skills: {len(parsed.get('skills', []))} found")
except json.JSONDecodeError as e:
    print(f"\n⚠️ JSON parsing error: {e}")

**LCEL Key Concepts:**

1. **Pipe Operator (|)**: Chains components together, output flows to next input
2. **PromptTemplate**: Reusable templates with variable substitution
3. **Invoke Pattern**: `chain.invoke({"var": value})` runs the pipeline
4. **Temperature=0**: Ensures consistent, deterministic outputs
5. **Modularity**: Each component is testable and reusable

---

# ✅ Question 6: Multi-Stage Pipeline Orchestration

**Question:** Build a complete resume screening pipeline that:
1. Extracts resume data (name, skills, etc.)
2. Compares candidate skills against required job skills
3. Generates a final report with scoring
4. Includes error handling at each stage
5. Returns all intermediate results for debugging

In [ ]:
# Solution: Multi-Stage Pipeline Function

def process_resume_pipeline(resume_text, required_skills):
    """
    Complete resume screening pipeline with three stages:
    1. Extract resume data
    2. Analyze skill gaps
    3. Generate final report with scoring
    
    Args:
        resume_text (str): Raw text from resume
        required_skills (list): Skills required for the job
    
    Returns:
        tuple: (final_report, resume_data, skill_gap_analysis)
    """
    
    # ====== STAGE 1: Extract Resume Data ======
    print("🔍 Stage 1: Extracting resume data...")
    try:
        # Invoke extraction chain
        extract_result = extract_chain.invoke({"resume_text": resume_text})
        resume_json_text = clean_json_text(extract_result.content)
        resume_data = json.loads(resume_json_text)
        print(f"   ✅ Extracted: {resume_data.get('name', 'Unknown')}")
        print(f"   ✅ Skills found: {len(resume_data.get('skills', []))}")
    except Exception as e:
        print(f"   ❌ Extraction failed: {e}")
        return None, None, None
    
    # ====== STAGE 2: Skill Gap Analysis ======
    print("\n📊 Stage 2: Analyzing skill gaps...")
    try:
        candidate_skills = resume_data.get('skills', [])
        
        # Create skill gap prompt
        skill_gap_prompt = PromptTemplate(
            input_variables=["candidate_skills", "required_skills"],
            template="""
Compare the candidate's skills against job requirements.

Candidate Skills: {candidate_skills}
Required Skills: {required_skills}

Return ONLY JSON with:
- missing_skills: array of skills the candidate lacks
- recommendations: object mapping each missing skill to a learning recommendation

Example format:
{{
  "missing_skills": ["Docker", "Kubernetes"],
  "recommendations": {{
    "Docker": "Complete Docker basics course and practice containerization",
    "Kubernetes": "Learn K8s fundamentals and deploy sample applications"
  }}
}}
"""
        )
        
        skill_gap_chain = skill_gap_prompt | llm
        gap_result = skill_gap_chain.invoke({
            "candidate_skills": json.dumps(candidate_skills),
            "required_skills": json.dumps(required_skills)
        })
        
        gap_json_text = clean_json_text(gap_result.content)
        skill_gap_data = json.loads(gap_json_text)
        print(f"   ✅ Missing skills: {len(skill_gap_data.get('missing_skills', []))}")
    except Exception as e:
        print(f"   ❌ Skill gap analysis failed: {e}")
        skill_gap_data = {"missing_skills": [], "recommendations": {}}
    
    # ====== STAGE 3: Generate Final Report ======
    print("\n📝 Stage 3: Generating final report...")
    try:
        final_prompt = PromptTemplate(
            input_variables=["candidate_json", "skill_gap_json"],
            template="""
Create a final hiring recommendation report.

Candidate Data:
{candidate_json}

Skill Gap Analysis:
{skill_gap_json}

Return ONLY JSON with all candidate fields PLUS:
- overall_score: integer 0-100 based on skill match and experience
- short_recommendation: brief hiring recommendation (1-2 sentences)
- missing_skills: array from skill gap analysis
"""
        )
        
        final_chain = final_prompt | llm
        final_result = final_chain.invoke({
            "candidate_json": json.dumps(resume_data, indent=2),
            "skill_gap_json": json.dumps(skill_gap_data, indent=2)
        })
        
        final_json_text = clean_json_text(final_result.content)
        final_report = json.loads(final_json_text)
        print(f"   ✅ Overall score: {final_report.get('overall_score', 'N/A')}/100")
        print(f"   ✅ Report generated successfully")
    except Exception as e:
        print(f"   ❌ Final report generation failed: {e}")
        final_report = None
    
    return final_report, resume_data, skill_gap_data


# Helper function to clean JSON from LLM output
def clean_json_text(text):
    """Remove code fences and clean JSON output."""
    if not isinstance(text, str):
        text = str(text)
    text = text.strip()
    # Remove ```json ... ``` or ``` ... ``` fences
    text = re.sub(r"```json\s*|```\s*", "", text)
    return text.strip()


print("✅ Pipeline function defined and ready to use!")

In [ ]:
# Test the complete pipeline

test_resume = """
SARAH JOHNSON
sarah.j@email.com | +1-555-8888

PROFESSIONAL SUMMARY
Data Engineer with 4 years of experience in building ETL pipelines and data warehouses.

TECHNICAL SKILLS
Python, SQL, Pandas, NumPy, Git, PostgreSQL, MongoDB

EDUCATION
B.S. in Computer Science, UCLA, 2017-2021, GPA: 3.6/4.0
"""

required_job_skills = [
    "Python", "SQL", "PySpark", "AWS", "Docker", 
    "Airflow", "Pandas", "Git", "ETL"
]

print("🚀 Running complete pipeline...\n")
print("=" * 70)

final, resume, gaps = process_resume_pipeline(test_resume, required_job_skills)

print("\n" + "=" * 70)
print("\n📊 FINAL REPORT:")
print(json.dumps(final, indent=2))

**Pipeline Best Practices:**

1. **Error Handling**: Wrap each stage in try/except blocks
2. **Graceful Degradation**: Provide fallback values when stages fail
3. **Logging**: Print progress messages for debugging
4. **Return Intermediates**: Return all stage outputs for inspection
5. **Modularity**: Each stage is independent and testable

---

# ✅ Question 7: Batch Processing & Data Analysis

**Question:** Process multiple resumes and create a ranked candidate list:
1. Process 3+ sample resumes through the pipeline
2. Collect all results in a list
3. Convert to a Pandas DataFrame
4. Sort by overall_score (descending)
5. Display top candidates with key metrics
6. Save results to CSV file

In [ ]:
# Solution: Batch Processing with Pandas Analysis

import pandas as pd

# Sample resumes for batch processing
sample_resumes = [
    {
        "filename": "candidate_1.pdf",
        "text": """
ALEX KUMAR
alex.kumar@techmail.com | (555) 111-2222

SUMMARY
Senior Data Engineer with 8 years building production ML pipelines.

SKILLS
Python, SQL, PySpark, AWS (S3, EMR, Glue), Docker, Kubernetes, Airflow, 
TensorFlow, Pandas, NumPy, Git, CI/CD

EDUCATION
M.S. Computer Science, Stanford University, 2013-2015, GPA: 3.9/4.0
"""
    },
    {
        "filename": "candidate_2.pdf",
        "text": """
MARIA GARCIA
maria.g@email.com | (555) 333-4444

SUMMARY
Data Analyst transitioning to Data Engineering, 2 years experience.

SKILLS
Python, SQL, Pandas, Excel, Tableau, Git, Basic AWS

EDUCATION
B.S. Statistics, University of Texas, 2019-2023, GPA: 3.5/4.0
"""
    },
    {
        "filename": "candidate_3.pdf",
        "text": """
JAMES WONG
j.wong@devmail.com | (555) 555-6666

SUMMARY
Data Engineer with 5 years experience in cloud data platforms.

SKILLS
Python, SQL, PySpark, GCP (BigQuery, Dataflow), Apache Beam, 
Docker, Airflow, dbt, Pandas, Git

EDUCATION
B.Eng. Computer Engineering, Georgia Tech, 2015-2019, GPA: 3.7/4.0
"""
    }
]

# Job requirements
job_skills = [
    "Python", "SQL", "PySpark", "AWS", "Docker", 
    "Airflow", "Pandas", "Git", "ETL", "Cloud Platforms"
]

print("🔄 Processing batch of resumes...\n")
print("=" * 70)

# Collect all results
all_results = []

for idx, resume in enumerate(sample_resumes, 1):
    print(f"\n📄 Processing {resume['filename']} ({idx}/{len(sample_resumes)})")
    print("-" * 70)
    
    # Run pipeline
    final_report, resume_data, skill_gaps = process_resume_pipeline(
        resume['text'], 
        job_skills
    )
    
    if final_report:
        # Add filename to report
        final_report['filename'] = resume['filename']
        all_results.append(final_report)
    
    print("-" * 70)

print("\n" + "=" * 70)
print(f"\n✅ Batch processing complete! {len(all_results)} resumes processed.")

In [ ]:
# Create DataFrame and analyze results

# Convert to DataFrame
df = pd.DataFrame(all_results)

# Select key columns for ranking
ranking_columns = [
    'filename', 'name', 'email', 'overall_score', 
    'short_recommendation', 'missing_skills'
]

# Create ranking view (only columns that exist)
available_cols = [col for col in ranking_columns if col in df.columns]
df_ranking = df[available_cols].copy()

# Sort by score (descending)
df_ranking = df_ranking.sort_values('overall_score', ascending=False)

# Reset index for clean display
df_ranking.reset_index(drop=True, inplace=True)
df_ranking.index = df_ranking.index + 1  # Start from 1

print("\n🏆 CANDIDATE RANKING\n")
print("=" * 100)
print(df_ranking.to_string())
print("=" * 100)

# Statistics
print("\n📊 STATISTICS:")
print(f"   Average Score: {df['overall_score'].mean():.1f}")
print(f"   Highest Score: {df['overall_score'].max()}")
print(f"   Lowest Score: {df['overall_score'].min()}")
print(f"   Score Std Dev: {df['overall_score'].std():.1f}")

In [ ]:
# Save results to CSV

output_file = "candidate_ranking_results.csv"

df_ranking.to_csv(output_file, index=True, index_label='Rank')

print(f"\n💾 Results saved to: {output_file}")
print(f"   Total candidates: {len(df_ranking)}")
print(f"   Columns saved: {len(df_ranking.columns)}")
print("\n✅ Batch processing and analysis complete!")

**Data Analysis Best Practices:**

1. **Pandas Integration**: Convert results to DataFrame for analysis
2. **Column Selection**: Choose relevant columns for clarity
3. **Sorting**: Rank by key metrics (score, experience, etc.)
4. **Statistics**: Calculate aggregate metrics for insights
5. **Export**: Save to CSV for sharing and reporting

---

# 🎯 Bonus Challenge: Advanced Extensions

## Challenge 1: Add DOCX Support
Modify the PDF loader to also handle `.docx` files using `python-docx` library.

## Challenge 2: Weighted Scoring
Implement a weighted scoring system where:
- Technical skills = 40%
- Experience years = 30%
- Education level = 20%
- Soft skills = 10%

## Challenge 3: Visualization
Create visualizations using matplotlib or plotly:
- Score distribution histogram
- Skills gap comparison chart
- Experience vs. Score scatter plot

## Challenge 4: Custom Job Descriptions
Build a function that:
- Takes a job description as input
- Extracts required skills automatically
- Determines seniority level
- Adjusts scoring criteria accordingly

---

# 📚 Summary & Key Takeaways

## What You've Learned:

### 1. **Environment & Security**
- Loading API keys from `.env` files
- Proper error handling for missing credentials
- Security best practices

### 2. **Document Processing**
- PyPDFLoader for PDF extraction
- Multi-page document handling
- Text cleaning with regex

### 3. **Schema Design**
- Pydantic BaseModel for validation
- Nested schemas for complex data
- Optional fields and default values

### 4. **LangChain LCEL**
- Pipe operator (|) for chaining
- PromptTemplate for reusable prompts
- Temperature settings for consistency

### 5. **Pipeline Orchestration**
- Multi-stage processing
- Error handling and graceful degradation
- Intermediate result tracking

### 6. **Data Analysis**
- Batch processing multiple documents
- Pandas for data manipulation
- Ranking and sorting candidates
- CSV export for reporting

## Production Considerations:
- ✅ Error handling at every stage
- ✅ Clean JSON output parsing
- ✅ Logging and debugging support
- ✅ Scalable batch processing
- ✅ Structured output validation

---

## 🎉 Congratulations!
You've completed the AI Resume Analyzer homework and learned production-grade AI application development!